<a href="https://colab.research.google.com/github/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/colab/04_chain_dimensions_across_ped.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the Google Colab version of this notebook. A copy with all its output is [in the SOURSOP repository](https://github.com/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/04_chain_dimensions_across_ped.ipynb).

# 4. Chain dimensions across PED

Because PED is large, it can be used as a dataset in its own right. Here we take a set of proteins that were all modelled the same way, AF-CALVADOS ensemble predictions (100 conformers each), and look at how their size scales with chain length. Using one method throughout keeps method-to-method differences ([notebook 3](https://colab.research.google.com/github/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/colab/03_one_protein_many_methods.ipynb)) out of the comparison.

For each protein we compute the mean radius of gyration and the apparent scaling exponent, fit Rg = R0 N^ν across proteins, and check SOURSOP's numbers against the values PED reports for the same ensembles.

In [ ]:
# Colab does not come with SOURSOP, so install it first. This uses uv, which
# is much faster than pip (and is itself installed first if it is missing);
# --system installs into Colab's own Python rather than a virtual environment.
# The PED module is new in SOURSOP 2.0.8; if that version is not on PyPI yet,
# this installs the latest version from GitHub instead.
!command -v uv >/dev/null || pip install -q uv
!uv pip install --system -q "soursop>=2.0.8" 2>/dev/null || uv pip install --system -q "git+https://github.com/holehouse-lab/soursop.git"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from soursop import ped

# Figure style: Arial throughout, text kept editable in Illustrator, no grid
# lines, and the xmgrace colour order (black, red, blue, green, purple).
# Colab does not have Arial, so there figures fall back to Liberation Sans
# (metrically identical to Arial) or matplotlib's default, DejaVu Sans
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.grid": False,
    "axes.prop_cycle": plt.cycler(color=["black", "red", "blue", "green", "purple"]),
})

# Downloaded ensembles are kept here and reused, so each is only fetched once
# per session (Colab deletes them when the session ends)
CACHE = "ped_cache"

from soursop.ssexceptions import SSException

## Choose the proteins

A single search returns the candidates, with no downloads. We keep single-ensemble, single-chain entries, and skip long chains to keep the run time (and the downloads) modest.

In [ ]:
N_PROTEINS = 30        # how many proteins to analyse
MAX_RESIDUES = 400     # skip very long chains

candidates = [e for e in ped.search_entries("AF-CALVADOS", max_results=80) if len(e.ensembles) == 1]
print(f"{len(candidates)} single-ensemble AF-CALVADOS entries to choose from")

## Load and analyse

Search results don't include chain lengths, but each entry's full record does, via the chain sequence. It also holds PED's own statistics, which we use at the end. So for each candidate we fetch the record first (a small request), and only download the ensemble if the protein is short enough. Downloads go into the cache, so re-running is much faster.

The scaling-exponent fit needs a chain of at least about 25 residues; shorter proteins would be kept for the Rg analysis but get no exponent.

In [ ]:
rows = []
for entry in candidates:
    if len(rows) >= N_PROTEINS:
        break
    # the full record gives the chain length without downloading anything,
    # so long or multi-chain proteins are skipped before a large download
    summary = ped.get_entry(entry.entry_id).ensembles[0]
    if len(summary.chains) != 1 or len(summary.chains[0].sequence or "") > MAX_RESIDUES:
        continue
    traj = summary.load(cache_dir=CACHE)
    if len(traj.proteinTrajectoryList) != 1:
        continue
    p = traj.proteinTrajectoryList[0]
    n = len(p.resid_with_CA)
    try:
        nu = p.get_scaling_exponent(mode="CA", verbose=False)[0]
    except SSException:
        nu = np.nan
    rows.append({
        "id": summary.identifier,
        "name": entry.title.replace("AF-CALVADOS ensemble prediction", "").replace(" of ", "").strip() or "(unnamed)",
        "n": n,
        "rg": p.get_radius_of_gyration().mean(),
        "nu": nu,
        "ped": summary.ped_statistics,
    })

print(f"{'ensemble':14s}{'N':>5s}{'<Rg> (A)':>10s}{'nu':>7s}  protein")
for r in sorted(rows, key=lambda r: r["n"]):
    print(f"{r['id']:14s}{r['n']:5d}{r['rg']:10.1f}{r['nu']:7.2f}  {r['name'][:45]}")

## Size against chain length

AF-CALVADOS predicts ensembles for whole proteins, and some of these proteins are folded. A folded protein's inter-residue distances stop growing beyond the size of the domain, so its fitted exponent is small, zero or even negative, and is not a meaningful polymer exponent. We therefore split the proteins into **expanded** chains (ν ≥ 0.45) and **compact** ones, and fit the power law Rg = R0 N^ν, a straight line on log-log axes, to the expanded chains only.

In [ ]:
from matplotlib.ticker import NullFormatter, ScalarFormatter

n = np.array([r["n"] for r in rows], dtype=float)
rg = np.array([r["rg"] for r in rows])
nu_each = np.array([r["nu"] for r in rows])
expanded = np.isfinite(nu_each) & (nu_each >= 0.45)
print(f"{expanded.sum()} expanded chains, {(~expanded).sum()} compact")

slope, intercept = np.polyfit(np.log(n[expanded]), np.log(rg[expanded]), 1)
print(f"Expanded chains: Rg = {np.exp(intercept):.2f} * N^{slope:.2f}")

fig, ax = plt.subplots(figsize=(3.6, 2.9))
ax.plot(n[expanded], rg[expanded], "o", color="black", ms=4, label="expanded")
ax.plot(n[~expanded], rg[~expanded], "o", mfc="none", mec="red", ms=4, label="compact")
grid = np.linspace(n[expanded].min(), n[expanded].max(), 100)
ax.plot(grid, np.exp(intercept) * grid**slope, color="black", lw=1)
ax.set_xscale("log")
ax.set_yscale("log")
for axis, ticks in ((ax.xaxis, [50, 100, 200, 400]), (ax.yaxis, [10, 20, 30, 50])):
    axis.set_major_formatter(ScalarFormatter())
    axis.set_minor_formatter(NullFormatter())
    axis.set_ticks(ticks)
ax.set_xlabel("Chain length N (residues)")
ax.set_ylabel("Mean radius of gyration (Å)")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
nu = np.array([r["nu"] for r in rows])
nu = nu[np.isfinite(nu)]
print(f"Per-protein exponents: median {np.median(nu):.2f}, range {nu.min():.2f} to {nu.max():.2f} ({len(nu)} proteins)")

fig, ax = plt.subplots(figsize=(3.4, 2.6))
ax.hist(nu, bins=np.arange(-0.2, 0.75, 0.05), color="black", histtype="step")
ax.axvline(0.59, color="black", ls="--", lw=0.8)
ax.set_xlabel("Apparent scaling exponent ν")
ax.set_ylabel("Proteins (count)")
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

The dashed line marks a self-avoiding random coil (ν ≈ 0.59). The expanded chains cluster between about 0.52 and 0.64, around the self-avoiding value, as expected for disordered proteins. The compact proteins trail off towards zero: for folded or largely folded proteins the exponent stops being a polymer property at all.

## Check against PED's own analysis

PED reports its own Rg and Flory exponent for each ensemble (`ped_statistics`, in nanometres). We already have them from the entry records fetched above.

In [ ]:
ped_rg, ped_nu, our_rg, our_nu = [], [], [], []
for r in rows:
    stats = r["ped"]
    if "rg_mean" in stats:
        ped_rg.append(10 * stats["rg_mean"])
        our_rg.append(r["rg"])
    if "flory_exponent" in stats and np.isfinite(r["nu"]):
        ped_nu.append(stats["flory_exponent"])
        our_nu.append(r["nu"])

ped_rg, our_rg = np.array(ped_rg), np.array(our_rg)
print(f"Rg: largest difference between SOURSOP and PED = {np.max(np.abs(ped_rg - our_rg)):.3f} A ({len(ped_rg)} ensembles)")

fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.9))
axes[0].plot(ped_rg, our_rg, "o", color="black", ms=4)
lims = [min(ped_rg.min(), our_rg.min()), max(ped_rg.max(), our_rg.max())]
axes[0].plot(lims, lims, color="red", lw=0.8)
axes[0].set_xlabel("PED mean Rg (Å)")
axes[0].set_ylabel("SOURSOP mean Rg (Å)")

axes[1].plot(ped_nu, our_nu, "o", color="black", ms=4)
lims = [min(min(ped_nu), min(our_nu)), max(max(ped_nu), max(our_nu))]
axes[1].plot(lims, lims, color="red", lw=0.8)
axes[1].set_xlabel("PED Flory exponent")
axes[1].set_ylabel("SOURSOP scaling exponent ν")
plt.tight_layout()
plt.show()

The radii of gyration agree exactly: both are the all-atom Rg of the same coordinates. The scaling exponents agree for the expanded chains, and much less well for the compact ones. That is expected, because the two fits make different choices: SOURSOP fits the RMS internal-scaling profile over a trimmed range of separations (`inter_residue_min`, `end_effect`), which PED does not do. Treat apparent exponents as method-dependent summaries rather than exact quantities, and compare them only when they were computed the same way.